In [1]:
import os
os.getcwd()

'/Users/kati/Desktop/agentic-rag-homework'

In [2]:
!pip install onnxruntime tokenizers numpy tqdm minsearch gitsource huggingface-hub


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [3]:
import onnxruntime
import tokenizers
import numpy
import tqdm
import minsearch
import gitsource

print("Все библиотеки установлены ✅")

Все библиотеки установлены ✅


In [5]:
!curl -O https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed/download.py
!curl -O https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed/embedder.py

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1376  100  1376    0     0   6896      0 --:--:-- --:--:-- --:--:--  6880
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1520  100  1520    0     0   9637      0 --:--:-- --:--:-- --:--:--  9681


In [6]:
!ls

download.py            llm-zoomcamp-hw1.ipynb
embedder.py            llm-zoomcamp-hw2.ipynb


In [7]:
!python download.py

tokenizer.json: 712kB [00:00, 107MB/s]
  saved models/Xenova/all-MiniLM-L6-v2/tokenizer.json
onnx/model.onnx: 100%|█████████████████████| 90.4M/90.4M [00:36<00:00, 2.47MB/s]
  saved models/Xenova/all-MiniLM-L6-v2/model.onnx


In [8]:
from embedder import Embedder

embedder = Embedder()

In [9]:
query = "How does approximate nearest neighbor search work?"

v = embedder.encode(query)

In [10]:
type(v)

numpy.ndarray

In [13]:
len(v)

384

In [14]:
v[:10]

array([-0.02058204, -0.01404591,  0.03029944, -0.05403782,  0.07187814,
       -0.02795373, -0.05030937, -0.0127217 ,  0.0408208 , -0.02600372])

### **Q1**

In [15]:
v[0]

np.float64(-0.020582036807885073)

### **Q2**

In [16]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [17]:
len(documents)

72

In [18]:
documents[0].keys()

dict_keys(['content', 'filename'])

In [19]:
documents[0]["filename"]

'01-agentic-rag/lessons/01-intro.md'

In [20]:
doc = next(
    d for d in documents
    if d["filename"] == "02-vector-search/lessons/07-sqlitesearch-vector.md"
)

doc["filename"]

'02-vector-search/lessons/07-sqlitesearch-vector.md'

In [21]:
doc_vector = embedder.encode(doc["content"])

similarity = doc_vector @ v
similarity

np.float64(0.3610702803026059)

### **Q3**

In [22]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

len(chunks)

295

In [23]:
chunks[0].keys()

dict_keys(['start', 'content', 'filename'])

In [24]:
chunks[0]["filename"]

'01-agentic-rag/lessons/01-intro.md'

In [25]:
chunks[0]["content"][:500]

"# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we'll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack"

In [26]:
chunk_texts = [chunk["content"] for chunk in chunks]

X = embedder.encode_batch(chunk_texts)

type(X), X.shape

(numpy.ndarray, (295, 384))

In [27]:
scores = X.dot(v)

scores.shape

(295,)

In [28]:
import numpy as np

best_idx = np.argmax(scores)
best_chunk = chunks[best_idx]

best_idx, scores[best_idx], best_chunk["filename"]

(np.int64(94),
 np.float64(0.648901732433228),
 '02-vector-search/lessons/07-sqlitesearch-vector.md')

### **Q3**

In [30]:
from minsearch import VectorSearch

vector_index = VectorSearch(
    keyword_fields=["filename", "start"]
)

vector_index.fit(X, chunks)

In [31]:
query_q4 = "What metric do we use to evaluate a search engine?"

v_q4 = embedder.encode(query_q4)

results_q4 = vector_index.search(
    query_vector=v_q4,
    num_results=5
)

[(r["filename"], r["start"]) for r in results_q4]

[('04-evaluation/lessons/05-search-metrics.md', 0),
 ('04-evaluation/lessons/01-intro.md', 0),
 ('01-agentic-rag/lessons/05-search.md', 0),
 ('04-evaluation/lessons/01-intro.md', 2000),
 ('04-evaluation/lessons/15-next-steps.md', 0)]

### **Q5**

In [32]:
from minsearch import Index

text_index = Index(
    text_fields=["content"],
    keyword_fields=["filename", "start"]
)

text_index.fit(chunks)

In [33]:
query_q5 = "How do I store vectors in PostgreSQL?"

text_results = text_index.search(
    query=query_q5,
    num_results=5
)

[(r["filename"], r["start"]) for r in text_results]

[('02-vector-search/lessons/02-embeddings.md', 4000),
 ('03-orchestration/lessons/05-rag.md', 1000),
 ('02-vector-search/lessons/01-intro.md', 0),
 ('03-orchestration/lessons/05-rag.md', 0),
 ('02-vector-search/lessons/01-intro.md', 1000)]

In [36]:
query_q5 = "How do I store vectors in PostgreSQL?"

v_q5 = embedder.encode(query_q5)

results_q5 = vector_index.search(
    query_vector=v_q5,
    num_results=5
)

[(r["filename"], r["start"]) for r in results_q5]

[('02-vector-search/lessons/08-pgvector.md', 0),
 ('02-vector-search/lessons/08-pgvector.md', 3000),
 ('03-orchestration/lessons/05-rag.md', 2000),
 ('02-vector-search/lessons/08-pgvector.md', 1000),
 ('02-vector-search/lessons/08-pgvector.md', 2000)]

### **Q6**

In [38]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [39]:
query_q6 = "How do I give the model access to tools?"

v_q6 = embedder.encode(query_q6)

vector_results_q6 = vector_index.search(
    query_vector=v_q6,
    num_results=5
)

text_results_q6 = text_index.search(
    query=query_q6,
    num_results=5
)

In [40]:
results_q6 = rrf([vector_results_q6, text_results_q6])

[(r["filename"], r["start"]) for r in results_q6]

[('01-agentic-rag/lessons/13-function-calling.md', 4000),
 ('01-agentic-rag/lessons/01-intro.md', 2000),
 ('01-agentic-rag/lessons/14-agentic-loop.md', 0),
 ('04-evaluation/lessons/02-ground-truth.md', 1000),
 ('01-agentic-rag/lessons/16-other-frameworks.md', 0)]